## Project Overview
This notebook implements a complete **Nurse Shift Scheduling System** using constraint satisfaction techniques and heuristic-based scheduling algorithms.

### Main Objectives
- Generate valid nurse schedules
- Respect hard scheduling constraints
- Improve fairness between nurses
- Demonstrate optimization and AI-related scheduling techniques

### Notebook Structure
1. Imports and setup
2. Dataset preparation
3. Configuration system
4. Core scheduling model
5. Hard constraints
6. Helper utilities
7. CSP and backtracking algorithms
8. Propagation and optimization
9. Execution and testing

### Learning Goals
This notebook is useful for understanding:
- Constraint Satisfaction Problems (CSP)
- Backtracking algorithms
- Heuristic optimization
- Real-world scheduling systems
- Python software organization

# Nurse Shift Scheduling System
## Self-Contained Notebook — Run on Any PC

All code is defined directly in cells below — no imports between modules, no external files.

Run cells **top to bottom**, or use **Kernel → Restart & Run All**.

## Imports

This cell imports the required libraries and dependencies.

In [ ]:
from __future__ import annotations
import argparse, ast, copy, csv, math, os, random, statistics, sys, time
from collections import deque
from dataclasses import dataclass, field
from datetime import datetime, date
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Set, Tuple


## Dataset
25 nurses — embedded directly, no external file needed.

This cell displays scheduling results and outputs.

In [ ]:
CSV_DATA = '''ID,LastName,FirstName,DateOfBirth,Age,Gender,seniority,day_off1,day_off2
1,BEBOUDI,IMAN,23/09/2002,23,femme,junior,2,3
2,BEDBOUDI,NDJATE,10/06/1980,45,femme,junior,1,5
3,BADRAOUI,KHEIREDDINE,29/10/1980,45,homme,junior,2,3
4,BERABEZ,HOUDRA,31/05/1985,40,femme,senior,6,5
5,BERRAHIL,ELHADI,10/01/1966,60,homme,senior,6,7
6,BERRAHIL,ABDELKARIM,13/07/1970,55,homme,junior,7,4
7,BRADAI,IMEN,02/08/1990,36,femme,senior,7,1
8,BRANES,HOUSSEM EDDINE,19/10/1995,30,homme,junior,5,2
9,BERBEGUE,SAMIRA,30/01/1997,29,femme,junior,3,4
10,BERBAGUE,KHOULOUD,07/07/1996,29,femme,senior,3,6
11,BERDIOM,MAROUA,01/07/1994,31,femme,senior,2,7
12,BERHI,CHAIMA,30/10/2002,23,femme,junior,4,7
13,BERROGTANE,MIANEL,14/04/1990,36,femme,junior,5,4
14,BARKOUK,LAILA,30/03/1999,27,femme,junior,6,2
15,BERKANI,ASIA,21/06/1981,44,femme,junior,7,4
16,BERKOUS,MERIEM,13/01/2003,23,femme,senior,3,5
17,BERNAMDANE,INA,05/09/1990,35,femme,junior,6,3
18,BEROUR,ROUMAISSA,21/11/1996,29,femme,junior,2,3
19,BRISEKH,YASMINA,24/10/1972,53,femme,junior,4,5
20,BRIZAKH,RACHID,05/08/1970,55,homme,senior,4,6
21,BESSIKEUR,BADREDDINE,19/11/1995,30,homme,junior,6,7
22,BSIKER,NESRINE,07/10/1995,30,femme,junior,6,1
23,IBRIR,NABILA,09/07/1981,44,femme,junior,7,2
24,ERHAILI,AMMAR,12/06/1966,59,homme,junior,4,3
25,AKTOUF,KHAIER EDINE,28/07/1965,60,homme,junior,7,4
'''

os.makedirs('data', exist_ok=True)
with open('data/data_ai_project.csv', 'w', encoding='utf-8') as f:
    f.write(CSV_DATA)

CSV_PATH = 'data/data_ai_project.csv'
print('CSV ready:', CSV_PATH)


---
## Code

### config
Configuration — shifts, coverage, constraints

This cell implements part of the nurse scheduling logic.

In [ ]:
# =============================================================================
# config.py — Central configuration for the Nurse Scheduling System
# =============================================================================

NUM_DAYS = 28
NUM_NURSES = 25

DAYS_OF_WEEK = list(range(1, 8))

SENIORITY_SENIOR = 'senior'
SENIORITY_JUNIOR = 'junior'

CSV_COLUMNS = {
    'id':         'ID',
    'last_name':  'LastName',
    'first_name': 'FirstName',
    'dob':        'DateOfBirth',
    'age':        'Age',
    'gender':     'Gender',
    'seniority':  'seniority',
    'day_off1':   'day_off1',
    'day_off2':   'day_off2',
}

# Five shift types: D=Day, L=Late, N=Night, O=requested Off, F=Free (unassigned)
# O and F both give 0 hours but are semantically different:
#   O = nurse's right (requested), F = scheduling artefact (not requested)
SHIFTS = {
    'D': {'name': 'Day',   'start': 8,    'end': 16,   'hours': 8},
    'L': {'name': 'Late',  'start': 16,   'end': 24,   'hours': 8},
    'N': {'name': 'Night', 'start': 0,    'end': 8,    'hours': 8},
    'O': {'name': 'Off',   'start': None, 'end': None, 'hours': 0},
    'F': {'name': 'Free',  'start': None, 'end': None, 'hours': 0},
}

WORKING_SHIFTS    = {'D', 'L', 'N'}
NON_WORKING_SHIFTS = {'O', 'F'}
ALL_SHIFT_CODES   = {'D', 'L', 'N', 'O', 'F'}

# N→L is forbidden: Night ends 08:00, Late starts 16:00 = exactly 8h (need strictly more)
FORBIDDEN_TRANSITIONS = {
    ('D', 'L'),
    ('L', 'N'),
    ('N', 'D'),
    ('N', 'L'),
}

COVERAGE = {
    'D': {'required': 7, 'min_senior': 1},
    'L': {'required': 7, 'min_senior': 1},
    'N': {'required': 4, 'min_senior': 1},
}

HARD = {
    'max_consecutive_work_days':   5,
    'max_consecutive_rest_days':   3,
    'max_monthly_hours':         170,
    'min_monthly_hours':          120,
    'max_consecutive_night_shifts': 2,
    'one_shift_per_day':          True,
}

WEIGHTS = {
    'day_off_first':  15,
    'day_off_second':  7,
    'consec_night':   10,
    'night_variance':  5,
    'hours_variance':  2,
    'reward_day_off_first': 10,
    'reward_day_off_second': 4
}

DAY_OFF_PRIORITY = {
    0: 'day_off_first',
    1: 'day_off_second',
}

DAY_OFF_MIN_PER_DAY = 3
DAY_OFF_MAX_PER_DAY = 4
DAY_OFF_RANGE       = (1, 7)

### 2. Core Model definition (`model.py`)
This cell defines the core `Nurse` dataclass and the `Schedule` state which represents a full 28-day schedule matrix for all 25 nurses. 

**Key Components:**
- `Nurse`: Data encapsulation for each nurse's attributes (Seniority, requests for days off, age, ID). 
- `Schedule`: Grid represented by a Dictionary indexed by `(day, nurse_id)`. Offers built-in statistics computations such as total hours and night shift queries.

This cell imports the required libraries and dependencies.

In [ ]:
from __future__ import annotations

import csv
import copy
import random
from dataclasses import dataclass, field
from datetime import datetime, date
from typing import Dict, List, Optional, Tuple




@dataclass
class Nurse:
   
    nurse_id:   int
    last_name:  str
    first_name: str
    dob:        date
    age:        int
    gender:     str
    seniority:  str                # 'senior' or 'junior'
    day_off1:   int                
    day_off2:   int                



    @property
    def is_senior(self) -> bool:
        """True if this nurse is classified as 'senior'."""
        return self.seniority == 'senior'

    @property
    def full_name(self) -> str:
        return f"{self.first_name} {self.last_name}"

    @property
    def preferred_days_off(self) -> Tuple[int, int]:
        """Return (primary, secondary) preferred days-off as a tuple."""
        return (self.day_off1, self.day_off2)

    def __repr__(self) -> str:
        badge = "SR" if self.is_senior else "JR"
        return f"Nurse({self.nurse_id}, {self.full_name}, {badge})"



#independent function to load the data :
#this function will open the csv file read up to limit row and transform each row to a nusrse object and return at the end an array of nurses (size=limit)
def load_nurses_from_csv(filepath: str, limit: int = NUM_NURSES) -> List[Nurse]:
   
    nurses: List[Nurse] = []
    col = CSV_COLUMNS  # this dict is defined in the config 

    with open(filepath, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            if i >= limit:
                break

            # Parse date of birth (format in CSV: DD/MM/YYYY)
            try:
                dob = datetime.strptime(row[col['dob']], "%d/%m/%Y").date()
            except ValueError:
                dob = date(1990, 1, 1)  

            nurse = Nurse(
                nurse_id=int(row[col['id']]),
                last_name=row[col['last_name']].strip(),
                first_name=row[col['first_name']].strip(),
                dob=dob,
                age=int(row[col['age']]),
                gender=row[col['gender']].strip(),
                seniority=row[col['seniority']].strip(),
                day_off1=int(row[col['day_off1']]),
                day_off2=int(row[col['day_off2']]),
            )
            nurses.append(nurse)

    if len(nurses) < limit:
        raise ValueError(
            f"CSV only has {len(nurses)} rows but {limit} nurses are required."
        )

    return nurses


class Schedule:
    

    def __init__(self, nurses: List[Nurse]):

        if len(nurses) != NUM_NURSES:
            raise ValueError(
                f"Schedule requires exactly {NUM_NURSES} nurses, got {len(nurses)}."
            )

        # Store nurses indexed by nurse_id for fast lookup
        self.nurses: List[Nurse] = nurses
        self.nurse_by_id: Dict[int, Nurse] = {n.nurse_id: n for n in nurses}

        # Initialize the grid: every cell is 'F' (free / not yet assigned)
        self.grid: Dict[Tuple[int, int], str] = {
            (day, nurse.nurse_id): 'F'
            for day in range(1, NUM_DAYS + 1)
            for nurse in nurses
        }

   
    #Return the shift code assigned to (day, nurse_id)
    def get(self, day: int, nurse_id: int) -> str:
        
        return self.grid[(day, nurse_id)]

    def set(self, day: int, nurse_id: int, shift: str) -> None:     #Assign a shift to (day, nurse_id).
       
        if shift not in ALL_SHIFT_CODES:
            raise ValueError(f"Invalid shift code '{shift}'. Must be one of {ALL_SHIFT_CODES}.")
        self.grid[(day, nurse_id)] = shift

    
    def get_nurse_schedule(self, nurse_id: int) -> List[str]:
    
        #Return the full 28-day schedule for one nurse as a list

    
        return [self.grid[(day, nurse_id)] for day in range(1, NUM_DAYS + 1)]

    def get_day_assignments(self, day: int) -> Dict[int, str]:
       "this will return the assignment of 1 day as  a dict[key=nurseId, value= shift_type ] "
       return {nid: self.grid[(day, nid)] for nid in self.nurse_by_id}

    def nurses_on_shift(self, day: int, shift: str) -> List[Nurse]: # Return all Nurse objects assigned to "shift" on "day"
        
        return [
            self.nurse_by_id[nid]
            for nid in self.nurse_by_id
            if self.grid[(day, nid)] == shift
        ]

    #statistics :

    def total_hours(self, nurse_id: int) -> int:
        "Return the total hours worked by nurse_id over the 28-day period"
        return sum(
            SHIFTS[self.grid[(day, nurse_id)]]['hours']
            for day in range(1, NUM_DAYS + 1)
        )

    def night_shift_count(self, nurse_id: int) -> int:
        "Return the number of night shifts ('N') assigned to nurse_id"
        return sum(
            1 for day in range(1, NUM_DAYS + 1)
            if self.grid[(day, nurse_id)] == 'N'
        )

    def working_days(self, nurse_id: int) -> List[int]:
        "Return list of days (1-28) on which nurse_id is on a working shift"
        return [
            day for day in range(1, NUM_DAYS + 1)
            if self.grid[(day, nurse_id)] in WORKING_SHIFTS
        ]

    
    def copy(self) -> "Schedule":
        "this will create a copy of the schedule "
        new_sched = Schedule(self.nurses)
        new_sched.grid = dict(self.grid)  
        return new_sched

    

    def summary(self) -> str:
        "for Debugging: this will return a summary about the scheduling , total hours , night shifts number  "
        lines = ["Day:     " + "  ".join(f"{d:2}" for d in range(1, NUM_DAYS + 1))]
        for nurse in self.nurses:
            row = self.get_nurse_schedule(nurse.nurse_id)
            badge = "SR" if nurse.is_senior else "JR"
            shifts_str = "  ".join(row)
            hours = self.total_hours(nurse.nurse_id)
            nights = self.night_shift_count(nurse.nurse_id)
            lines.append(
                f"[{badge}] {nurse.full_name:<25} {shifts_str}  | {hours}h | {nights}N"
            )
        return "\n".join(lines)

    def __repr__(self) -> str:
        return f"Schedule(nurses={NUM_NURSES}, days={NUM_DAYS})"

### 3. Hard Constraints Logic (`hard_constraints.py`)
This module enforces the **Absolute Rules** of our Nurse Scheduling Problem. 

If any of these conditions are broken, the schedule is deemed **infeasible** and mathematically rejected.
**Constraints Monitored:**
- **H1:** One shift maximum per day per staff member.
- **H2:** Prohibition of toxic transitions (e.g. Night Shift to subsequent Morning Shift leaving insufficient rest).
- **H3:** Limit of 5 consecutive active working days.
- **H4:** Cap on sequential resting days (to avoid over-gapping).
- **H5:** Compliance with working hour boundaries (120 - 170 minimum/maximum range).
- **H6:** Minimum and exact hospital staffing coverage for D/L/N per day, factoring in Seniority ratios.
- **H7:** Absolute block on >2 consecutive night shifts.

The function `is_valid_assignment` enables lightning-fast checks to assess if isolated hypothetical swaps would break any constraints, critical for Local Search algorithms like SA and Tabu.

This cell imports the required libraries and dependencies.

In [ ]:
# =============================================================================
# core/hard_constraints.py — Hard constraint checkers for the Nurse Scheduling System
# =============================================================================
# All checkers operate on a Schedule object and return a list of violation
# strings (empty list = no violations for that constraint).
#
# Public API:
#   check_all_hard(schedule)         → List[str]  (all violations combined)
#   is_valid_assignment(schedule, day, nurse_id, shift) → bool
#       (used by SA / Tabu before accepting a swap)
# =============================================================================

from __future__ import annotations

from typing import List



# ---------------------------------------------------------------------------
# H1 — One shift per day
# ---------------------------------------------------------------------------

def check_one_shift_per_day(schedule: Schedule) -> List[str]:
    """
    Each nurse must be assigned exactly one shift code per day.
    The Schedule grid guarantees a single entry per (day, nurse_id) cell,
    so this check validates that the code is a legal shift code.
    (Belt-and-suspenders guard — model.set() already enforces this.)
    """
    violations: List[str] = []
    for nurse in schedule.nurses:
        for day in range(1, NUM_DAYS + 1):
            shift = schedule.get(day, nurse.nurse_id)
            if shift not in SHIFTS:
                violations.append(
                    f"[H1] Nurse {nurse.nurse_id} day {day}: "
                    f"invalid shift code '{shift}'"
                )
    return violations


# ---------------------------------------------------------------------------
# H2 — Forbidden shift transitions (minimum rest between shifts)
# ---------------------------------------------------------------------------

def check_forbidden_transitions(schedule: Schedule) -> List[str]:
    """
    Certain back-to-back shift pairs are forbidden because they leave fewer
    than the required 8 h rest between the end of one shift and the start of
    the next:
        D→L  (Day ends 16:00, Late starts 16:00 — 0 h gap)
        L→N  (Late ends 24:00, Night starts 00:00 — 0 h gap)
        N→D  (Night ends 08:00, Day starts 08:00 — 0 h gap)
        N→L  (Night ends 08:00, Late starts 16:00 — exactly 8 h, need > 8 h)

    Day boundaries wrap: day 28 → day 1 is NOT checked (end of period).
    """
    violations: List[str] = []
    for nurse in schedule.nurses:
        nid = nurse.nurse_id
        for day in range(1, NUM_DAYS):          # day → day+1
            today = schedule.get(day, nid)
            tomorrow = schedule.get(day + 1, nid)
            if (today, tomorrow) in FORBIDDEN_TRANSITIONS:
                violations.append(
                    f"[H2] Nurse {nid} forbidden transition "
                    f"'{today}'→'{tomorrow}' on days {day}→{day + 1}"
                )
    return violations


# ---------------------------------------------------------------------------
# H3 — Maximum 5 consecutive working days
# ---------------------------------------------------------------------------

def check_max_consecutive_work_days(schedule: Schedule) -> List[str]:
    """
    No nurse may work more than HARD['max_consecutive_work_days'] (= 5)
    working days in a row (D, L, or N).
    """
    max_consec = HARD['max_consecutive_work_days']
    violations: List[str] = []

    for nurse in schedule.nurses:
        nid = nurse.nurse_id
        streak = 0
        for day in range(1, NUM_DAYS + 1):
            if schedule.get(day, nid) in WORKING_SHIFTS:
                streak += 1
                if streak > max_consec:
                    violations.append(
                        f"[H3] Nurse {nid}: {streak} consecutive working days "
                        f"ending on day {day} (max {max_consec})"
                    )
            else:
                streak = 0
    return violations


# ---------------------------------------------------------------------------
# H4 — Maximum 3 consecutive rest days
# ---------------------------------------------------------------------------

def check_max_consecutive_rest_days(schedule: Schedule) -> List[str]:
    """
    No nurse may have more than HARD['max_consecutive_rest_days'] (= 3)
    consecutive non-working days (O or F).
    Equivalently: max 72 h of not-working time in a row.
    """
    max_consec = HARD['max_consecutive_rest_days']
    violations: List[str] = []

    for nurse in schedule.nurses:
        nid = nurse.nurse_id
        streak = 0
        for day in range(1, NUM_DAYS + 1):
            if schedule.get(day, nid) in NON_WORKING_SHIFTS:
                streak += 1
                if streak > max_consec:
                    violations.append(
                        f"[H4] Nurse {nid}: {streak} consecutive rest days "
                        f"ending on day {day} (max {max_consec})"
                    )
            else:
                streak = 0
    return violations


# ---------------------------------------------------------------------------
# H5 — Monthly hours: min 80 h, max 160 h
# ---------------------------------------------------------------------------

def check_monthly_hours(schedule: Schedule) -> List[str]:
    """
    Each nurse's total working hours over the 28-day period must be within
    [HARD['min_monthly_hours'], HARD['max_monthly_hours']] = [80, 160].
    """
    min_h = HARD['min_monthly_hours']
    max_h = HARD['max_monthly_hours']
    violations: List[str] = []

    for nurse in schedule.nurses:
        nid = nurse.nurse_id
        total = schedule.total_hours(nid)
        if total < min_h:
            violations.append(
                f"[H5] Nurse {nid}: only {total} h worked (min {min_h} h)"
            )
        elif total > max_h:
            violations.append(
                f"[H5] Nurse {nid}: {total} h worked (max {max_h} h)"
            )
    return violations


# ---------------------------------------------------------------------------
# H6 — Daily coverage: required headcount + at least one senior per shift
# ---------------------------------------------------------------------------

def check_daily_coverage(schedule: Schedule) -> List[str]:
    """
    For every day and every working shift type (D, L, N):
      • exactly COVERAGE[shift]['required'] nurses must be assigned
      • at least COVERAGE[shift]['min_senior'] of them must be senior

    Note: 'exactly' is used here; the coverage numbers in config are
    requirements, not upper bounds. A generator that produces fewer workers
    is just as wrong as one that produces more.
    """
    violations: List[str] = []

    for day in range(1, NUM_DAYS + 1):
        assignments = schedule.get_day_assignments(day)    # {nurse_id: shift}

        for shift_code, req in COVERAGE.items():
            required = req['required']
            min_senior = req['min_senior']

            # Nurses assigned to this shift today
            assigned_nurses = [
                schedule.nurse_by_id[nid]
                for nid, s in assignments.items()
                if s == shift_code
            ]
            count = len(assigned_nurses)
            senior_count = sum(1 for n in assigned_nurses if n.is_senior)

            if count != required:
                violations.append(
                    f"[H6] Day {day} shift '{shift_code}': "
                    f"{count} nurses assigned (required {required})"
                )
            if senior_count < min_senior:
                violations.append(
                    f"[H6] Day {day} shift '{shift_code}': "
                    f"{senior_count} senior(s) assigned (min {min_senior})"
                )
    return violations


# ---------------------------------------------------------------------------
# H7 — Max 2 consecutive night shifts
# ---------------------------------------------------------------------------

def check_max_consecutive_night_shifts(schedule: Schedule) -> List[str]:
    """
    No nurse may work more than HARD['max_consecutive_night_shifts'] (= 2)
    night shifts ('N') in a row. This is listed as a soft constraint in the
    project spec but also enforced as a hard limit by the config.
    """
    max_consec = HARD['max_consecutive_night_shifts']
    violations: List[str] = []

    for nurse in schedule.nurses:
        nid = nurse.nurse_id
        streak = 0
        for day in range(1, NUM_DAYS + 1):
            if schedule.get(day, nid) == 'N':
                streak += 1
                if streak > max_consec:
                    violations.append(
                        f"[H7] Nurse {nid}: {streak} consecutive night shifts "
                        f"ending on day {day} (max {max_consec})"
                    )
            else:
                streak = 0
    return violations


# ---------------------------------------------------------------------------
# Master checker
# ---------------------------------------------------------------------------

def check_all_hard(schedule: Schedule) -> List[str]:
    """
    Run every hard-constraint checker and return the combined list of
    violation messages.  An empty list means the schedule is feasible.
    """
    violations: List[str] = []
    violations.extend(check_one_shift_per_day(schedule))
    violations.extend(check_forbidden_transitions(schedule))
    violations.extend(check_max_consecutive_work_days(schedule))
    violations.extend(check_max_consecutive_rest_days(schedule))
    violations.extend(check_monthly_hours(schedule))
    violations.extend(check_daily_coverage(schedule))
    violations.extend(check_max_consecutive_night_shifts(schedule))
    return violations


# ---------------------------------------------------------------------------
# Single-assignment validator (used by SA / Tabu before accepting swaps)
# ---------------------------------------------------------------------------

def is_valid_assignment(schedule: Schedule, day: int, nurse_id: int, shift: str) -> bool:
    """
    Check whether tentatively assigning `shift` to `nurse_id` on `day` would
    violate any hard constraint, WITHOUT modifying the schedule.

    This is called by the local-search algorithms before committing a swap, so
    it must be fast and must not mutate the schedule.

    Checks performed (in short-circuit order):
      1. Forbidden transition with the previous day's shift.
      2. Forbidden transition with the next day's shift.
      3. Consecutive working-day streak (5-day cap).
      4. Consecutive rest-day streak (3-day cap) — only if shift is non-working.
      5. Consecutive night-shift streak (2-night cap).
      6. Monthly hours bounds after the proposed change.
      7. Daily coverage feasibility (headcount + senior requirement).

    Returns True iff all checks pass.
    """
    current_shift = schedule.get(day, nurse_id)

    # --- Transition checks ---
    prev_shift = schedule.get(day - 1, nurse_id) if day > 1 else None
    next_shift = schedule.get(day + 1, nurse_id) if day < NUM_DAYS else None

    if prev_shift is not None and (prev_shift, shift) in FORBIDDEN_TRANSITIONS:
        return False
    if next_shift is not None and (shift, next_shift) in FORBIDDEN_TRANSITIONS:
        return False

    # --- Consecutive working days ---
    if shift in WORKING_SHIFTS:
        streak_before = _count_streak_before(schedule, day, nurse_id, WORKING_SHIFTS)
        streak_after = _count_streak_after(schedule, day, nurse_id, WORKING_SHIFTS)
        if streak_before + 1 + streak_after > HARD['max_consecutive_work_days']:
            return False

    # --- Consecutive rest days ---
    if shift in NON_WORKING_SHIFTS:
        streak_before = _count_streak_before(schedule, day, nurse_id, NON_WORKING_SHIFTS)
        streak_after = _count_streak_after(schedule, day, nurse_id, NON_WORKING_SHIFTS)
        if streak_before + 1 + streak_after > HARD['max_consecutive_rest_days']:
            return False

    # --- Consecutive night shifts ---
    if shift == 'N':
        streak_before = _count_streak_before(schedule, day, nurse_id, {'N'})
        streak_after = _count_streak_after(schedule, day, nurse_id, {'N'})
        if streak_before + 1 + streak_after > HARD['max_consecutive_night_shifts']:
            return False

    # --- Monthly hours ---
    delta_hours = SHIFTS[shift]['hours'] - SHIFTS[current_shift]['hours']
    new_total = schedule.total_hours(nurse_id) + delta_hours
    if not (HARD['min_monthly_hours'] <= new_total <= HARD['max_monthly_hours']):
        return False

    # --- Daily coverage (check the affected day only) ---
    if not _coverage_ok_after_change(schedule, day, nurse_id, current_shift, shift):
        return False

    return True


# ---------------------------------------------------------------------------
# Internal helpers
# ---------------------------------------------------------------------------

def _count_streak_before(
    schedule: Schedule,
    day: int,
    nurse_id: int,
    shift_set: set,
) -> int:
    """Count how many consecutive days immediately before `day` have a shift in `shift_set`."""
    count = 0
    d = day - 1
    while d >= 1 and schedule.get(d, nurse_id) in shift_set:
        count += 1
        d -= 1
    return count


def _count_streak_after(
    schedule: Schedule,
    day: int,
    nurse_id: int,
    shift_set: set,
) -> int:
    """Count how many consecutive days immediately after `day` have a shift in `shift_set`."""
    count = 0
    d = day + 1
    while d <= NUM_DAYS and schedule.get(d, nurse_id) in shift_set:
        count += 1
        d += 1
    return count


def _coverage_ok_after_change(
    schedule: Schedule,
    day: int,
    nurse_id: int,
    old_shift: str,
    new_shift: str,
) -> bool:
    """
    Simulate replacing `old_shift` with `new_shift` for `nurse_id` on `day`
    and verify that coverage constraints are still met.

    For working shifts we require exactly the required number of nurses AND
    at least one senior — so we verify both directions (the shift being
    vacated and the shift being taken).
    """
    nurse = schedule.nurse_by_id[nurse_id]
    assignments = schedule.get_day_assignments(day)

    # Let's count once to avoid iterating all assignments repeatedly
    counts: dict[str, int] = {}
    seniors: dict[str, int] = {}
    for s in ALL_SHIFT_CODES:
        counts[s] = 0
        seniors[s] = 0

    for nid, s in assignments.items():
        if s in counts:
            counts[s] += 1
            if schedule.nurse_by_id[nid].is_senior:
                seniors[s] += 1

    for shift_code, req in COVERAGE.items():
        required = req['required']
        min_senior = req['min_senior']

        current_count = counts.get(shift_code, 0)
        current_seniors = seniors.get(shift_code, 0)

        # Adjust for the proposed change
        new_count = current_count
        new_seniors = current_seniors

        if old_shift == shift_code:
            new_count -= 1
            if nurse.is_senior:
                new_seniors -= 1

        if new_shift == shift_code:
            new_count += 1
            if nurse.is_senior:
                new_seniors += 1

        # Validate only working shifts (O / F have no coverage requirement)
        if shift_code in WORKING_SHIFTS:
            if new_count != required:
                return False
            if new_seniors < min_senior:
                return False

    return True

### 4. Search Filter & Helper (`generator_helpers.py`)
This section extracts helper methods that filter lists of eligible nurses for explicit day allocations. 
Functions like `_consecutive_work` and `_consecutive_rest` scan the matrix timeline backwards to calculate real-time streaks.

This cell imports the required libraries and dependencies.

In [ ]:
"""Helper functions for the schedule generator."""
from __future__ import annotations
import random
from typing import List, Set
def _eligible(
    schedule: Schedule,
    nurses: List[Nurse],
    day: int,
    shift_code: str,
    already_assigned: Set[int],
) -> List[Nurse]:
    """
    Filters and returns a list of nurses who are capable of taking the requested shift
    on the current day without breaking any hard constraints.
    
    Hard constraints checked:
      - Has not already been assigned to another shift today.
      - Cell is free ("F").
      - Follows forbidden transitions (e.g. Morning after Night shift).
      - Remains within maximum consecutive working days limits.
      - Remains within maximum consecutive night shifts limit.
      - Remains within the maximum monthly hours cap.
    """
    result = []
    for nurse in nurses:
        nid = nurse.nurse_id
        if nid in already_assigned:
            continue
        if day > 0 and schedule.get(day, nid) not in ("F",):
            continue
        
        if day > 1:
            yesterday = schedule.get(day - 1, nid)
            if yesterday in WORKING_SHIFTS:
                if (yesterday, shift_code) in FORBIDDEN_TRANSITIONS:
                    continue
        if _consecutive_work(schedule, nid, day) >= HARD["max_consecutive_work_days"]:
            continue
        if shift_code == "N":
            if _consecutive_nights(schedule, nid, day) >= HARD["max_consecutive_night_shifts"]:
                continue
        hours_so_far = schedule.total_hours(nid)
        shift_hours = SHIFTS[shift_code]["hours"]
        if hours_so_far + shift_hours > HARD["max_monthly_hours"]:
            continue
            
        result.append(nurse)
    return result

def _consecutive_work(schedule: Schedule, nurse_id: int, day: int) -> int:
    """
    Counts the number of consecutive working days a nurse has had leading up to (and not including) the current day.
    """
    count = 0
    for d in range(day - 1, 0, -1):
        s = schedule.get(d, nurse_id)
        if s in WORKING_SHIFTS:
            count += 1
        else:
            break
    return count

def _consecutive_rest(schedule: Schedule, nurse_id: int, day: int) -> int:
    """
    Counts the number of consecutive rest days a nurse has had leading up to (and not including) the current day.
    """
    count = 0
    for d in range(day - 1, 0, -1):
        s = schedule.get(d, nurse_id)
        if s in NON_WORKING_SHIFTS:
            count += 1
        else:
            break
    return count

def _consecutive_nights(schedule: Schedule, nurse_id: int, day: int) -> int:
    """
    Counts the number of consecutive night shifts a nurse has had leading up to (and not including) the current day.
    """
    count = 0
    for d in range(day - 1, 0, -1):
        s = schedule.get(d, nurse_id)
        if s == "N":
            count += 1
        else:
            break
    return count

### 5. CSP Random Generator (`generator.py`)
This solver establishes the initial **Feasible Solution State**.
It operates using a chronological day-by-day greedy construction logic, combined with **Backtracking**. 

If the schedule runs into a dead-end loop on Day 18 (i.e. zero eligible staff remaining for Night coverage), it recursively wipes recent days and re-attempts random branches until the final 28-day state satisfies **ALL Hard Constraints**. Soft constraints/penalties are disregarded during this phase, creating a strictly compliant but likely sub-optimal schedule that serves as our foundation.

This cell imports the required libraries and dependencies.

In [ ]:
"""Fast initial schedule generator with hard-constraint checks. (don't worry about soft constraints here, we'll fix those later)"""
from __future__ import annotations
import random
from typing import List, Set
MAX_BACKTRACK = 5000
RANDOM_SEED = None

def _meets_min_hours(schedule: Schedule) -> bool:
	min_hours = HARD["min_monthly_hours"]
	for nurse in schedule.nurses:
		# Verify each nurse reaches the minimum monthly hours.
		total = schedule.total_hours(nurse.nurse_id)
		if total < min_hours:
			return False
	return True

#main function that generate the schedule
def generate_schedule(nurses: List[Nurse], seed: int | None = RANDOM_SEED) -> Schedule:
	if seed is not None:
		random.seed(seed)
	schedule = Schedule(nurses)
	day = 1
	backtrack_count = 0
	while day <= NUM_DAYS:
		# Try to fill a single day; backtrack if coverage fails.
		success = _fill_day(schedule, nurses, day)
		if success:
			day += 1
		else:
			backtrack_count += 1
			# lightweight progress logging to help diagnose failures
			if backtrack_count % 50 == 0:
				print(f"[generator] backtracks={backtrack_count} at day={day}")
			if backtrack_count > MAX_BACKTRACK:
				raise RuntimeError(
					"Generator failed to build a feasible schedule after "
					f"{backtrack_count} backtracks (limit {MAX_BACKTRACK}) at day {day}. "
					"Try increasing MAX_BACKTRACK further, changing the seed, or check your constraints."
				)
			# here we clear the current day and the previous day to undo any bad assignments, then step back one day to retry(here we should use backtracking)
			_clear_day(schedule, nurses, day)
			if day > 1:
				_clear_day(schedule, nurses, day - 1)
				day -= 1
	# Ensure the final schedule meets the min hours requirement.
	if not _meets_min_hours(schedule):
		raise RuntimeError(
			"Generator produced a schedule that fails min monthly hours. "
			"Try increasing MAX_BACKTRACK or relaxing constraints."
		)
	return schedule

def _fill_day(schedule: Schedule, nurses: List[Nurse], day: int) -> bool:
	assigned_today: Set[int] = set()
	for shift_code in ("D", "L", "N"):
		required = COVERAGE[shift_code]["required"]
		min_senior = COVERAGE[shift_code]["min_senior"]
		eligible = _eligible(schedule, nurses, day, shift_code, assigned_today)
		# Split eligibility to guarantee at least one senior per shift.
		seniors = [n for n in eligible if n.is_senior]
		juniors = [n for n in eligible if not n.is_senior]
		_ = juniors
		if len(eligible) < required:
			return False
		if len(seniors) < min_senior:
			return False
		# Pick one senior, then fill the rest randomly from remaining eligible.
		chosen_seniors = random.sample(seniors, min_senior)
		remaining_pool = [n for n in eligible if n not in chosen_seniors]
		still_needed = required - min_senior
		if len(remaining_pool) < still_needed:
			return False
		chosen_fill = random.sample(remaining_pool, still_needed)
		chosen = chosen_seniors + chosen_fill
		for nurse in chosen:
			schedule.set(day, nurse.nurse_id, shift_code)
			assigned_today.add(nurse.nurse_id)
	for nurse in nurses:
		if nurse.nurse_id not in assigned_today:
			# Do not exceed max consecutive rest days.
			if _consecutive_rest(schedule, nurse.nurse_id, day) >= HARD["max_consecutive_rest_days"]:
				_clear_day(schedule, nurses, day)
				return False
			schedule.set(day, nurse.nurse_id, "O")
	return True

#this will be removed later, it is just a helper to clear a day when we need to backtrack 
def _clear_day(schedule: Schedule, nurses: List[Nurse], day: int) -> None:
	for nurse in nurses:
		schedule.set(day, nurse.nurse_id, "F")

### 6. Accelerated CSP Algorithm (`backtracking.py`)
An alternative heuristic-enhanced Backtracking solver designed to replace randomized backtracking. 

This introduces "smart ordering" variable choices:
- Assigns highly-constrained shifts first.
- Prioritizes Nurse staff nearest to maximum resting bounds or those with fewer alternative compatibilities.
- Reduces infinite looping trees in CSP.

This cell imports the required libraries and dependencies.

In [ ]:
from __future__ import annotations

import random
import time
from typing import Dict, List, Optional, Set, Tuple


_ASSIGNABLE = {'D', 'L', 'N', 'O'}


# ---------------------------------------------------------------------------
# Relaxed validity check for incremental construction
# ---------------------------------------------------------------------------
def _is_valid_for_bt(schedule, day, nurse_id, shift):
    """Check if assigning shift to nurse on day is valid during construction."""
    if day > 1:
        prev = schedule.get(day - 1, nurse_id)
        if prev != 'F' and (prev, shift) in FORBIDDEN_TRANSITIONS:
            return False
    if day < NUM_DAYS:
        nxt = schedule.get(day + 1, nurse_id)
        if nxt != 'F' and (shift, nxt) in FORBIDDEN_TRANSITIONS:
            return False
    if shift in WORKING_SHIFTS:
        streak = 1
        d = day - 1
        while d >= 1 and schedule.get(d, nurse_id) in WORKING_SHIFTS:
            streak += 1; d -= 1
        if streak > HARD['max_consecutive_work_days']:
            return False
    if shift == 'O':
        streak = 1
        d = day - 1
        while d >= 1 and schedule.get(d, nurse_id) == 'O':
            streak += 1; d -= 1
        if streak > HARD['max_consecutive_rest_days']:
            return False
    if shift == 'N':
        streak = 1
        d = day - 1
        while d >= 1 and schedule.get(d, nurse_id) == 'N':
            streak += 1; d -= 1
        if streak > HARD['max_consecutive_night_shifts']:
            return False
    current = schedule.get(day, nurse_id)
    delta = SHIFTS[shift]['hours'] - SHIFTS[current]['hours']
    if schedule.total_hours(nurse_id) + delta > HARD['max_monthly_hours']:
        return False
    return True


# ---------------------------------------------------------------------------
# Fill one day — dynamic shift ordering + smart nurse selection
# ---------------------------------------------------------------------------
def _fill_day(schedule, nurses, day):
    """Assign all nurses for one day. Returns True on success."""
    assigned: Set[int] = set()
    shifts_left = list(COVERAGE.keys())  # ['D', 'L', 'N']

    # Process shifts in order of tightest constraint (fewest excess eligible)
    while shifts_left:
        best_shift = None
        best_slack = float('inf')
        best_eligible = []

        for sc in shifts_left:
            elig = [n for n in nurses
                    if n.nurse_id not in assigned
                    and _is_valid_for_bt(schedule, day, n.nurse_id, sc)]
            slack = len(elig) - COVERAGE[sc]['required']
            if slack < best_slack:
                best_slack = slack
                best_shift = sc
                best_eligible = elig

        if best_slack < 0:
            for nid in assigned:
                schedule.set(day, nid, 'F')
            return False

        sc = best_shift
        shifts_left.remove(sc)
        req = COVERAGE[sc]['required']
        min_sr = COVERAGE[sc]['min_senior']
        eligible = best_eligible

        seniors = [n for n in eligible if n.is_senior]
        juniors = [n for n in eligible if not n.is_senior]

        if len(seniors) < min_sr:
            for nid in assigned:
                schedule.set(day, nid, 'F')
            return False

        # Prefer nurses with fewer alternative shifts (save flexible ones)
        def _alt(n):
            return sum(1 for s2 in shifts_left
                       if _is_valid_for_bt(schedule, day, n.nurse_id, s2))

        def _work_streak(n):
            streak = 0
            d = day - 1
            while d >= 1 and schedule.get(d, n.nurse_id) in WORKING_SHIFTS:
                streak += 1; d -= 1
            return streak

        def _rest_streak(n):
            streak = 0
            d = day - 1
            while d >= 1 and schedule.get(d, n.nurse_id) == 'O':
                streak += 1; d -= 1
            return streak

        def _sort_key(n):
            # 1. Forced-work nurses first (approaching max rest)
            # 2. Low work-streak preferred (avoid bunching forced-rest days)
            # 3. Fewer alternatives first (lock constrained nurses early)
            # 4. Lower hours first (balance workload)
            return (-_rest_streak(n), _work_streak(n), _alt(n),
                    schedule.total_hours(n.nurse_id), random.random())

        seniors.sort(key=_sort_key)
        juniors.sort(key=_sort_key)

        chosen_sr = seniors[:min_sr]
        chosen_ids = {n.nurse_id for n in chosen_sr}
        rest_pool = [n for n in seniors[min_sr:]] + juniors
        rest_pool.sort(key=_sort_key)
        chosen_fill = rest_pool[:req - min_sr]
        chosen = chosen_sr + chosen_fill

        for n in chosen:
            schedule.set(day, n.nurse_id, sc)
            assigned.add(n.nurse_id)

    # Remaining nurses get 'O' or fallback
    for nurse in nurses:
        if nurse.nurse_id not in assigned:
            if _is_valid_for_bt(schedule, day, nurse.nurse_id, 'O'):
                schedule.set(day, nurse.nurse_id, 'O')
            else:
                placed = False
                for fb in ('D', 'L', 'N'):
                    cur = sum(1 for nid in assigned
                              if schedule.get(day, nid) == fb)
                    if (cur < COVERAGE[fb]['required']
                            and _is_valid_for_bt(schedule, day, nurse.nurse_id, fb)):
                        schedule.set(day, nurse.nurse_id, fb)
                        placed = True
                        break
                if not placed:
                    _clear_day(schedule, nurses, day)
                    return False
            assigned.add(nurse.nurse_id)
    return True


def _clear_day(schedule, nurses, day):
    for nurse in nurses:
        schedule.set(day, nurse.nurse_id, 'F')


# ---------------------------------------------------------------------------
# Public entry point
# ---------------------------------------------------------------------------
def run_backtracking(
    nurses: list,
    time_limit_seconds: float = 300.0,
) -> Optional[Schedule]:
    """
    Build a hard-constraint-valid schedule using day-level backtracking
    with randomised greedy assignment, dynamic shift ordering, and
    smart nurse selection heuristics.

    Parameters
    ----------
    nurses : list of Nurse
    time_limit_seconds : float

    Returns
    -------
    Schedule or None
    """
    MAX_RESTARTS = 50
    MAX_BT = 300
    start = time.time()

    for restart in range(MAX_RESTARTS):
        if time.time() - start > time_limit_seconds:
            break

        schedule = Schedule(nurses)
        day = 1
        bt = 0

        while day <= NUM_DAYS:
            if time.time() - start > time_limit_seconds:
                break
            if _fill_day(schedule, nurses, day):
                day += 1
            else:
                bt += 1
                if bt > MAX_BT:
                    break
                _clear_day(schedule, nurses, day)
                back = min(3, day - 1) if bt % 5 == 0 else min(1, day - 1)
                for _ in range(back):
                    if day > 1:
                        _clear_day(schedule, nurses, day - 1)
                        day -= 1

        if day > NUM_DAYS:
            violations = check_all_hard(schedule)
            if not violations:
                elapsed = time.time() - start
                print(f"[Backtracking] restart={restart} bt={bt} "
                      f"time={elapsed:.2f}s solved=True")
                return schedule

    elapsed = time.time() - start
    print(f"[Backtracking] time={elapsed:.2f}s solved=False")
    return None

### propagation
AC-3 forward checking

This cell imports the required libraries and dependencies.

In [ ]:
from __future__ import annotations
 
from collections import deque
from typing import Dict, List, Set, Tuple
 
 
Domains = Dict[Tuple[int, int], Set[str]]
 
 
def get_initial_domains(schedule: Schedule) -> Domains:
    domains: Domains = {}
    for nurse in schedule.nurses:
        nid = nurse.nurse_id
        for day in range(1, NUM_DAYS + 1):
            var = (nid, day)
            current = schedule.get(day, nid)
            if current != 'F':
                domains[var] = {current}
            else:
                candidates: Set[str] = set()
                for shift in ALL_SHIFT_CODES:
                    if is_valid_assignment(schedule, day, nid, shift):
                        candidates.add(shift)
                domains[var] = candidates
    return domains
 
 
def forward_check(
    schedule: Schedule,
    day: int,
    nurse_id: int,
    domains: Domains,
) -> bool:
    affected: List[Tuple[int, int]] = []
 
    for delta in (-1, +1):
        neighbour_day = day + delta
        if 1 <= neighbour_day <= NUM_DAYS:
            if schedule.get(neighbour_day, nurse_id) == 'F':
                affected.append((nurse_id, neighbour_day))
 
    for nurse in schedule.nurses:
        nid = nurse.nurse_id
        if nid != nurse_id and schedule.get(day, nid) == 'F':
            affected.append((nid, day))
 
    for var in affected:
        n_id, d = var
        valid: Set[str] = {
            shift for shift in domains.get(var, set())
            if is_valid_assignment(schedule, d, n_id, shift)
        }
        domains[var] = valid
        if not valid:
            return False
 
    return True
 
 
def revise_arc(
    var1: Tuple[int, int],
    var2: Tuple[int, int],
    domains: Domains,
    schedule: Schedule,
) -> bool:
    nurse1_id, day1 = var1
    nurse2_id, day2 = var2
    to_remove: List[str] = []
 
    for s1 in list(domains.get(var1, set())):
        temp = schedule.copy()
        temp.set(day1, nurse1_id, s1)
        has_support = any(
            is_valid_assignment(temp, day2, nurse2_id, s2)
            for s2 in domains.get(var2, set())
        )
        if not has_support:
            to_remove.append(s1)
 
    for s1 in to_remove:
        domains[var1].discard(s1)
 
    return bool(to_remove)
 
 
def ac3_propagate(schedule: Schedule, domains: Domains) -> bool:
    def _are_neighbours(v1: Tuple[int, int], v2: Tuple[int, int]) -> bool:
        n1, d1 = v1
        n2, d2 = v2
        if v1 == v2:
            return False
        if n1 == n2 and abs(d1 - d2) == 1:
            return True
        if d1 == d2 and n1 != n2:
            return True
        return False
 
    all_vars = list(domains.keys())
    queue: deque = deque(
        (v1, v2)
        for v1 in all_vars
        for v2 in all_vars
        if _are_neighbours(v1, v2)
    )
 
    while queue:
        var_i, var_j = queue.popleft()
        if revise_arc(var_i, var_j, domains, schedule):
            if not domains[var_i]:
                return False
            for var_k in all_vars:
                if var_k != var_j and _are_neighbours(var_k, var_i):
                    queue.append((var_k, var_i))
 
    return True

### 7. Fairness Heuristics (`fairness_metrics.py`)
A standalone calculation method that assesses global fairness amongst staff through variance computation. Evaluates variables such as:
1. Hourly balance disparities.
2. Evenly assigning night shifts across the roster.
3. Spread of Consecutive Night Shifts.

The variance scores will penalize schedules that assign too many dense burdens onto subset nurses.

This cell imports the required libraries and dependencies.

In [ ]:
import statistics
def compute_fairness_metrics(schedule: Schedule) -> dict:
    all_hours=[]
    all_night_shifts=[]
    all_max_consec_nights = []
    for nurse in schedule.nurse_by_id.values():
        nurse_id = nurse.nurse_id
        all_hours.append(schedule.total_hours(nurse_id))
        total_nights = 0
        current_streak = 0
        max_streak = 0
        for day in range(1, NUM_DAYS + 1):
            if schedule.get(day, nurse_id) == 'N':
                current_streak += 1
                total_nights += 1
            else:
                max_streak = max(max_streak, current_streak)
                current_streak = 0
        max_streak = max(max_streak, current_streak)
        all_max_consec_nights.append(max_streak)
        all_night_shifts.append(total_nights)

    hours_variance = statistics.variance(all_hours) if len(all_hours) > 1 else 0
    night_shift_variance = statistics.variance(all_night_shifts) if len(all_night_shifts) > 1 else 0
    max_consec_nights_variance = statistics.variance(all_max_consec_nights) if len(all_max_consec_nights) > 1 else 0
    return {
        "hours_variance": hours_variance,
        "night_shift_variance": night_shift_variance,
        "consec_nights_variance": max_consec_nights_variance
    }

### 8. Cost & Evaluation Calculation (`evaluation.py`)
The universal scoring function. 

Calculates standard penalty points (negative impact to algorithm fitness score) based on **Soft Constraints**:
1. Approving Primary/Secondary weekend-off requests.
2. Fairness calculation mapping (`compute_fairness_metrics`).
3. Night shift distribution.

*Notice*: All local-search algorithms aim to **Minimize** this returned score natively. Negative output implies a mathematically preferred schedule layout.

This cell implements functions used for scheduling operations.

In [ ]:
def evaluate_schedule(schedule: Schedule) -> float:
    score = 0.0

    requested_day_off_1={x: [] for x in range(1,8)}
    requested_day_off_2={x: [] for x in range(1,8)}
    for nurse in schedule.nurse_by_id.values():
        if nurse.day_off1:
            requested_day_off_1[nurse.day_off1].append(nurse.nurse_id)
        if nurse.day_off2:
            requested_day_off_2[nurse.day_off2].append(nurse.nurse_id)
    priority_day_off_1 = {}
    priority_day_off_2 = {}
    for day in range(1, 8):
        sorted_nurses_1 = sorted(requested_day_off_1[day], key=lambda nid: schedule.nurse_by_id[nid].seniority == 'senior', reverse=True)
        priority_day_off_1[day] = set(sorted_nurses_1)

        sorted_nurses_2 = sorted(requested_day_off_2[day], key=lambda nid: schedule.nurse_by_id[nid].seniority == 'senior', reverse=True)
        priority_day_off_2[day] = set(sorted_nurses_2)

    fairness = compute_fairness_metrics(schedule)
    for nurse in schedule.nurse_by_id.values():
        nurse_id = nurse.nurse_id

        prefered_dayoff1= nurse.day_off1
        prefered_dayoff2= nurse.day_off2
        conse_nights=0
        if nurse.seniority == 'senior':
            senyority_weight = 1.5
        else:
            senyority_weight = 1.0
        for start in [1,8,15,22]:
            if prefered_dayoff1 and nurse_id in priority_day_off_1[prefered_dayoff1]:
                the_day1to_verify_each_week = start + prefered_dayoff1 - 1
                if schedule.get(the_day1to_verify_each_week, nurse_id) in WORKING_SHIFTS:
                    score += senyority_weight*WEIGHTS['day_off_first']
                else:
                    score -= senyority_weight*WEIGHTS['reward_day_off_first']
            
            if prefered_dayoff2 and nurse_id in priority_day_off_2[prefered_dayoff2]:
                the_day2to_verify_each_week = start + prefered_dayoff2 - 1
                if schedule.get(the_day2to_verify_each_week, nurse_id) in WORKING_SHIFTS:
                    score += senyority_weight*WEIGHTS['day_off_second']
                else:
                    score -= senyority_weight*WEIGHTS['reward_day_off_second']

        for day in range(1, 29):
            if schedule.get(day, nurse_id) == 'N':
                conse_nights+=1
                if conse_nights > HARD['max_consecutive_night_shifts']:
                 score += (conse_nights-HARD['max_consecutive_night_shifts'])*WEIGHTS['consec_night']
            else:
                conse_nights=0


    score += fairness['night_shift_variance']*WEIGHTS['night_variance']
    score += fairness['hours_variance']*WEIGHTS['hours_variance']
    score += (fairness['consec_nights_variance'] * WEIGHTS['consec_night'])
    return score

### 9. State Manipulators & Neighbour Search (`generate_neighbours.py`)
These mathematical operations are required to generate slightly perturbed iterations of the schedule.
Used fundamentally by Simulated Annealing and Tabu Search.

- **Shift Reassignment**: Blindly mutate single cell shifts.
- **Nurse Swap**: Safest manipulation. Interchanges identical days between Nurse A & Nurse B. Total facility coverage attributes perfectly sustained.
- **Day Swap**: Nurse A interchanging Shift X with Shift Y on Day 1 and Day 10. Excellent at breaking negative Work/Rest streaks.

This cell imports the required libraries and dependencies.

In [ ]:
#for simulated annealing and greedy
from __future__ import annotations
import random
from typing import List, Tuple, Optional, Dict, Any




Move = Dict[str, Any]  
Neighbour = Tuple[Schedule, Move]


def _try_shift_reassignment(schedule: Schedule) -> Optional[Neighbour]:
    """
    Pick a random (nurse, day) cell and try replacing its shift with a
    different valid shift code.

    Returns (new_schedule, move) if a valid reassignment is found,
    else None.
    """
    nurses = schedule.nurses
    days   = list(range(1, NUM_DAYS + 1))
    shifts = list(ALL_SHIFT_CODES)

    # Shuffle to avoid always biasing toward nurse 1 / day 1
    random.shuffle(nurses)
    random.shuffle(days)

    for nurse in nurses:
        nid = nurse.nurse_id
        for day in days:
            current = schedule.get(day, nid)
            candidates = [s for s in shifts if s != current]
            random.shuffle(candidates)

            for new_shift in candidates:
                if is_valid_assignment(schedule, day, nid, new_shift):
                    new_schedule = schedule.copy()
                    new_schedule.set(day, nid, new_shift)
                    move = {
                        'type':      'shift_reassignment',
                        'nurse_id':  nid,
                        'day':       day,
                        'old_shift': current,
                        'new_shift': new_shift,
                    }
                    return new_schedule, move

    return None



def _try_nurse_swap(schedule: Schedule) -> Optional[Neighbour]:
    """
    Pick a random day, then pick two nurses assigned to different shifts
    and swap them.

    This move preserves total coverage counts perfectly — whatever nurse A
    had, nurse B takes, and vice versa — making it the safest move type.

    Returns (new_schedule, move) if a valid swap is found, else None.
    """
    days = list(range(1, NUM_DAYS + 1))
    random.shuffle(days)

    for day in days:
        assignments = schedule.get_day_assignments(day)  # {nid: shift}
        nurse_ids   = list(assignments.keys())
        random.shuffle(nurse_ids)

        # Try all pairs
        for i in range(len(nurse_ids)):
            for j in range(i + 1, len(nurse_ids)):
                nid_a = nurse_ids[i]
                nid_b = nurse_ids[j]
                shift_a = assignments[nid_a]
                shift_b = assignments[nid_b]

                # No point swapping identical shifts
                if shift_a == shift_b:
                    continue

                # Check both directions are valid
                # Note: we must check A getting B's shift AND B getting A's shift.
                # is_valid_assignment checks the *full* schedule including coverage,
                # so we simulate the first change then check the second on a temp copy.
                temp = schedule.copy()
                temp.set(day, nid_a, shift_b)

                if (is_valid_assignment(schedule, day, nid_a, shift_b) and
                        is_valid_assignment(temp, day, nid_b, shift_a)):

                    new_schedule = temp.copy()
                    new_schedule.set(day, nid_b, shift_a)

                    move = {
                        'type':     'nurse_swap',
                        'day':      day,
                        'nurse_a':  nid_a,
                        'shift_a':  shift_a,
                        'nurse_b':  nid_b,
                        'shift_b':  shift_b,
                    }
                    return new_schedule, move

    return None



def _try_day_swap(schedule: Schedule) -> Optional[Neighbour]:
    """
    Pick a random nurse, then pick two days with different shifts and swap
    those shifts.

    Great for breaking consecutive-work or consecutive-rest streaks:
    e.g. if nurse works days 1-6 in a row, swapping day 3 (work) with
    day 10 (rest) breaks the streak.

    Returns (new_schedule, move) if a valid swap is found, else None.
    """
    nurses = list(schedule.nurses)
    random.shuffle(nurses)

    days = list(range(1, NUM_DAYS + 1))

    for nurse in nurses:
        nid = nurse.nurse_id
        random.shuffle(days)

        for i in range(len(days)):
            for j in range(i + 1, len(days)):
                day_a = days[i]
                day_b = days[j]
                shift_a = schedule.get(day_a, nid)
                shift_b = schedule.get(day_b, nid)

                if shift_a == shift_b:
                    continue

                # Simulate day_a getting shift_b, then day_b getting shift_a
                temp = schedule.copy()
                temp.set(day_a, nid, shift_b)

                if (is_valid_assignment(schedule, day_a, nid, shift_b) and
                        is_valid_assignment(temp, day_b, nid, shift_a)):

                    new_schedule = temp.copy()
                    new_schedule.set(day_b, nid, shift_a)

                    move = {
                        'type':     'day_swap',
                        'nurse_id': nid,
                        'day_a':    day_a,
                        'shift_a':  shift_a,
                        'day_b':    day_b,
                        'shift_b':  shift_b,
                    }
                    return new_schedule, move

    return None




_MOVE_FUNCTIONS = [
    _try_shift_reassignment,
    _try_nurse_swap,
    _try_day_swap,
]

_MOVE_WEIGHTS = [1, 3, 2]   # nurse_swap is most reliable, so higher weight


def get_random_neighbour(
    schedule: Schedule,
    mode: str = 'weighted',
) -> Optional[Neighbour]:
    """
    Return a single random valid neighbour and its move descriptor.

    Parameters
    ----------
    schedule : Schedule
        The current schedule to perturb.
    mode : str
        'weighted'  — pick move type by _MOVE_WEIGHTS (default, best for SA)
        'uniform'   — pick move type uniformly at random
        'swap_only' — only use nurse_swap (safest, good for fine-tuning)

    Returns
    -------
    (new_schedule, move) or None if no valid neighbour was found.
    """
    if mode == 'swap_only':
        return _try_nurse_swap(schedule)

    if mode == 'uniform':
        fns = list(_MOVE_FUNCTIONS)
        random.shuffle(fns)
        for fn in fns:
            result = fn(schedule)
            if result is not None:
                return result
        return None

    # weighted — default
    fn = random.choices(_MOVE_FUNCTIONS, weights=_MOVE_WEIGHTS, k=1)[0]
    result = fn(schedule)
    if result is not None:
        return result

    # Fallback: try remaining move types
    for fn in _MOVE_FUNCTIONS:
        result = fn(schedule)
        if result is not None:
            return result

    return None


def generate_neighbours(
    schedule: Schedule,
    n_samples: int = 20,
    mode: str = 'weighted',
) -> List[Neighbour]:
    """
    Return up to `n_samples` distinct valid neighbours.

    Used by Tabu Search which evaluates a batch of neighbours per iteration
    and picks the best one (even if it's slightly worse than current).

    Parameters
    ----------
    schedule  : current schedule
    n_samples : how many neighbours to generate (default 20)
    mode      : same as get_random_neighbour

    Returns
    -------
    List of (new_schedule, move) pairs. May be shorter than n_samples
    if the search space is very constrained.
    """
    neighbours: List[Neighbour] = []
    seen_moves = set()   # deduplicate by move signature

    max_attempts = n_samples * 10   # avoid infinite loop in tight spaces
    attempts = 0

    while len(neighbours) < n_samples and attempts < max_attempts:
        attempts += 1
        result = get_random_neighbour(schedule, mode=mode)
        if result is None:
            continue

        _, move = result
        sig = _move_signature(move)
        if sig in seen_moves:
            continue

        seen_moves.add(sig)
        neighbours.append(result)

    return neighbours



def _move_signature(move: Move) -> tuple:
    """
    Produce a hashable key that uniquely identifies a move.
    Used to deduplicate neighbour lists.
    """
    t = move['type']
    if t == 'shift_reassignment':
        return (t, move['nurse_id'], move['day'], move['new_shift'])
    elif t == 'nurse_swap':
        a, b = sorted([move['nurse_a'], move['nurse_b']])
        return (t, move['day'], a, b)
    elif t == 'day_swap':
        da, db = sorted([move['day_a'], move['day_b']])
        return (t, move['nurse_id'], da, db)
    return tuple(move.items())

### 10. Greedy Search Algorithm (`greedy.py`)
Provides an alternative scheduling engine operating purely on greedy heuristics logic (Best choice evaluation recursively without backtracking consideration).
Constructs allocations based primarily on "scarcity", guaranteeing seniors shift allocations primarily and leveraging continuity logic.

This cell imports the required libraries and dependencies.

In [ ]:
from typing import List




def is_valid_for_greedy(schedule: Schedule, day: int, nurse_id: int, shift: str) -> bool:
    """
    Return True only when assigning `shift` on `day` to `nurse_id` violates
    NO hard constraint.  Soft constraints are intentionally ignored here.
    """

    # 1. Forbidden shift transitions (previous day → today)
    if day > 1:
        prev_shift = schedule.get(day - 1, nurse_id)
        if prev_shift is not None and (prev_shift, shift) in FORBIDDEN_TRANSITIONS:
            return False

    # 2. Max consecutive working days
    if shift in WORKING_SHIFTS:
        streak_before = _count_streak_before(schedule, day, nurse_id, WORKING_SHIFTS)
        if streak_before + 1 > HARD['max_consecutive_work_days']:
            return False

    # 3. Max consecutive rest days
    if shift in NON_WORKING_SHIFTS:
        streak_before = _count_streak_before(schedule, day, nurse_id, NON_WORKING_SHIFTS)
        if streak_before + 1 > HARD['max_consecutive_rest_days']:
            return False

    # 4. Max consecutive night shifts
    if shift == 'N':
        streak_before = _count_streak_before(schedule, day, nurse_id, {'N'})
        if streak_before + 1 > HARD['max_consecutive_night_shifts']:
            return False

   
    new_total = schedule.total_hours(nurse_id) + SHIFTS[shift]['hours']
    if new_total > HARD['max_monthly_hours']:
        return False

    return True



def _score_nurse(schedule: Schedule, day: int, day_of_week: int,
                 nurse, shift_type: str) -> tuple:
  
    # — Force nurses out of a long rest streak (soft nudge toward work)
    forced = 0
    if day > 3:
        rest_streak = 0
        for d in range(day - 1, max(0, day - 4), -1):
            if schedule.get(d, nurse.nurse_id) in NON_WORKING_SHIFTS:
                rest_streak += 1
            else:
                break
        if rest_streak >= 3:
            forced = -1000          # strongly prefer to give them a shift

    # — Prefer nurses who did NOT request this day off
    requested_off = 1 if (nurse.day_off1 == day_of_week
                          or nurse.day_off2 == day_of_week) else 0

    # — Prefer nurses with fewer accumulated hours (load balancing)
    hours = schedule.total_hours(nurse.nurse_id)

    # — Prefer nurses with shorter current work streaks (avoid burnout)
    work_streak = 0
    for d in range(day - 1, max(0, day - 6), -1):
        if schedule.get(d, nurse.nurse_id) in WORKING_SHIFTS:
            work_streak += 1
        else:
            break

    # — For nights, prefer nurses already on a night run (continuity)
    night_continuity = 0
    if shift_type == 'N' and day > 1 and schedule.get(day - 1, nurse.nurse_id) == 'N':
        night_continuity = -1       # negative = preferred

    return (forced, requested_off, hours, work_streak, night_continuity)


def generate_greedy_schedule(schedule: Schedule) -> Schedule:
   

    for day in range(1, NUM_DAYS + 1):
        day_of_week = (day - 1) % 7 + 1
        assigned_today: set[int] = set()

   
        working_shift_types = list(COVERAGE.keys())   # e.g. ['D', 'L', 'N']

        def _scarcity(st):
            return len([
                n for n in schedule.nurses
                if n.nurse_id not in assigned_today
                and is_valid_for_greedy(schedule, day, n.nurse_id, st)
            ])

        working_shift_types.sort(key=_scarcity)       # ascending: fewest first

       
        for shift_type in working_shift_types:
            req       = COVERAGE[shift_type]['required']
            min_senior = COVERAGE[shift_type]['min_senior']

            
            candidates = [
                n for n in schedule.nurses
                if n.nurse_id not in assigned_today
                and is_valid_for_greedy(schedule, day, n.nurse_id, shift_type)
            ]

            candidates.sort(
                key=lambda n: _score_nurse(schedule, day, day_of_week, n, shift_type)
            )

            chosen: list  = []
            chosen_ids: set[int] = set()

            #Fill senior quota first
            for n in candidates:
                if len(chosen) >= min_senior:
                    break
                if n.is_senior:
                    chosen.append(n)
                    chosen_ids.add(n.nurse_id)

            #Top up to required (any seniority)
            for n in candidates:
                if len(chosen) >= req:
                    break
                if n.nurse_id not in chosen_ids:
                    chosen.append(n)
                    chosen_ids.add(n.nurse_id)

            for n in chosen:
                schedule.set(day, n.nurse_id, shift_type)
                assigned_today.add(n.nurse_id)

       
        for nurse in schedule.nurses:
            if nurse.nurse_id in assigned_today:
                continue

            requested_off = (nurse.day_off1 == day_of_week
                             or nurse.day_off2 == day_of_week)
            preferred_rest = 'O' if requested_off else 'F'

            if is_valid_for_greedy(schedule, day, nurse.nurse_id, preferred_rest):
               
                schedule.set(day, nurse.nurse_id, preferred_rest)

            else:
               
                fallback_assigned = False

               
                fallback_shifts = sorted(
                    working_shift_types,
                    key=lambda st: sum(
                        1 for n in schedule.nurses
                        if schedule.get(day, n.nurse_id) == st
                    )                                 
                )

                for fallback_shift in fallback_shifts:
                    if is_valid_for_greedy(schedule, day, nurse.nurse_id, fallback_shift):
                        schedule.set(day, nurse.nurse_id, fallback_shift)
                        assigned_today.add(nurse.nurse_id)
                        fallback_assigned = True
                        break

                if not fallback_assigned:
                   
                    import warnings
                    warnings.warn(
                        f"[Greedy] No hard-feasible shift found for nurse "
                        f"{nurse.nurse_id} on day {day}. "
                        f"Assigning '{preferred_rest}' as emergency fallback. "
                        f"Check your HARD constraint configuration.",
                        RuntimeWarning,
                        stacklevel=2,
                    )
                    schedule.set(day, nurse.nurse_id, preferred_rest)

            assigned_today.add(nurse.nurse_id)

    return schedule

### 11. Simulated Annealing Optimization (`simulated_annealing.py`)
Applies the formal thermodynamic model of cooling logic to navigate local minimums inside the search space.

- **Inputs**: Accept a purely feasible mathematical model out of the Backtracking Generator.
- **Logic**: Evaluates a neighbor configuration. Instantly adapts superior neighbors. Sub-optimal neighbors can still be accepted using probability driven by current High Temperature properties.
- As temperature geometrically approaches zero via `cooling_rate`, the function gradually crystallizes towards a pure greedy hill-climber logic.
- Optionally applies automatic Heat Injection (`reheat_factor`) when stagnant to escape isolated optima valleys.

This cell imports the required libraries and dependencies.

In [ ]:


from __future__ import annotations

import math
import random
import time
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple


# ---------------------------------------------------------------------------
# Result container
# ---------------------------------------------------------------------------

@dataclass
class SAResult:
    """Everything you might want to inspect after a SA run."""
    best_schedule:     Schedule
    best_score:        float
    initial_score:     float
    final_score:       float
    iterations:        int
    accepted_moves:    int
    rejected_moves:    int
    skipped_moves:     int
    best_iteration:    int
    final_temperature: float
    elapsed_seconds:   float
    history:           List[Dict[str, Any]] = field(default_factory=list)

    def summary(self) -> str:
        lines = [
            "=== Simulated Annealing Result ===",
            f"  Initial score      : {self.initial_score:.2f}",
            f"  Final score        : {self.final_score:.2f}",
            f"  Best score         : {self.best_score:.2f}",
            f"  Best found at iter : {self.best_iteration}",
            f"  Total iterations   : {self.iterations}",
            f"  Accepted moves     : {self.accepted_moves}",
            f"  Rejected moves     : {self.rejected_moves}",
            f"  Skipped (invalid)  : {self.skipped_moves}",
            f"  Final temperature  : {self.final_temperature:.6f}",
            f"  Elapsed time       : {self.elapsed_seconds:.2f}s",
            f"  Improvement        : {self.initial_score - self.best_score:.2f}",
        ]
        return "\n".join(lines)


# ---------------------------------------------------------------------------
# Acceptance criterion
# ---------------------------------------------------------------------------

def _accept_worse_move(delta: float, temperature: float) -> bool:
    """
    Metropolis acceptance criterion.

    Accepts a worse move (delta > 0 = higher penalty = worse) with
    probability exp(−delta / temperature).

    At high T  → probability ≈ 1  → explore freely.
    At low T   → probability ≈ 0  → converge on local optimum.

    The exponent is clamped at 700 to avoid math.exp overflow on very
    large deltas combined with a still-warm temperature.
    """
    if temperature <= 0.0:
        return False
    exponent = min(delta / temperature, 700.0)
    return random.random() < math.exp(-exponent)


def _random_same_day_swap(schedule: Schedule) -> Optional[Tuple[Schedule, Dict[str, Any]]]:
    """Return a hard-feasible swap between two nurses on the same day."""
    day = random.randint(1, NUM_DAYS)
    nurses = list(schedule.nurses)
    random.shuffle(nurses)

    for idx, nurse_a in enumerate(nurses):
        for nurse_b in nurses[idx + 1:]:
            shift_a = schedule.get(day, nurse_a.nurse_id)
            shift_b = schedule.get(day, nurse_b.nurse_id)
            if shift_a == shift_b:
                continue

            candidate = schedule.copy()
            candidate.set(day, nurse_a.nurse_id, shift_b)
            candidate.set(day, nurse_b.nurse_id, shift_a)
            if check_all_hard(candidate):
                continue

            return candidate, {
                "type": "same_day_swap",
                "day": day,
                "nurse_a": nurse_a.nurse_id,
                "shift_a": shift_a,
                "nurse_b": nurse_b.nurse_id,
                "shift_b": shift_b,
            }

    return None


def _sample_best_candidate(
    current_schedule: Schedule,
    current_score: float,
    samples: int,
    neighbour_mode: str,
    enforce_candidate_feasible: bool,
) -> Optional[Tuple[Schedule, float, Dict[str, Any]]]:
    """
    Sample several feasible moves and return the best scored candidate.

    The shared neighbour generator provides a small diverse batch. SA then
    fills the rest with direct same-day swaps, because those preserve coverage
    and tend to change the soft score more often.
    """
    best: Optional[Tuple[Schedule, float, Dict[str, Any]]] = None
    useful_count = 0

    neighbours = generate_neighbours(
        current_schedule,
        n_samples=1,
        mode=neighbour_mode,
    )

    for _ in range(max(0, samples - len(neighbours))):
        neighbour = _random_same_day_swap(current_schedule)
        if neighbour is not None:
            neighbours.append(neighbour)

    for candidate_schedule, move in neighbours:
        if enforce_candidate_feasible and check_all_hard(candidate_schedule):
            continue

        candidate_score = evaluate_schedule(candidate_schedule)
        useful_count += 1
        if best is None or candidate_score < best[1]:
            best = (candidate_schedule, candidate_score, move)

    for _ in range(max(0, 3 - useful_count)):
        neighbour = _random_same_day_swap(current_schedule)
        if neighbour is None:
            continue

        candidate_schedule, move = neighbour
        if enforce_candidate_feasible and check_all_hard(candidate_schedule):
            continue

        candidate_score = evaluate_schedule(candidate_schedule)
        if best is None or candidate_score < best[1]:
            best = (candidate_schedule, candidate_score, move)

    return best


# ---------------------------------------------------------------------------
# Main SA function
# ---------------------------------------------------------------------------

def simulated_annealing(
    initial_schedule: Schedule,
    # --- Temperature schedule ---
    initial_temperature: float = 150.0,
    cooling_rate:        float = 0.995,
    min_temperature:     float = 0.01,
    # --- Iteration budget ---
    max_iterations: int = 15_000,
    # --- Reheat (escape local optima) ---
    reheat_enabled:  bool  = True,
    reheat_patience: int   = 500,
    reheat_factor:   float = 0.25,
    max_reheats:     int   = 5,
    # --- Neighbour generation ---
    neighbour_mode: str = "weighted",
    candidates_per_iteration: int = 8,
    # --- Reproducibility ---
    seed: Optional[int] = None,
    # --- Logging ---
    log_every: int  = 200,
    verbose:   bool = False,
    # --- Safety guards ---
    require_initial_feasible:   bool = True,
    enforce_candidate_feasible: bool = False,
) -> SAResult:
    """
    Improve a hard-feasible schedule using Simulated Annealing.

    The schedule MUST come from a feasibility-guaranteed source —
    specifically core.generator.generate_csp_schedule().  SA does not
    repair infeasible schedules; it only optimises the soft-constraint score
    while keeping all hard constraints satisfied.

    Parameters
    ----------
    initial_schedule : Schedule
        A feasible starting point from the CSP generator (core/generator.py).

    initial_temperature : float
        Starting temperature T₀.  Higher = more exploration early on.
        Typical range for this problem: 50–300.

    cooling_rate : float in (0, 1)
        Geometric cooling: T ← T × rate each iteration.
        Typical: 0.990–0.999.  Smaller = faster cooling = less exploration.

    min_temperature : float
        SA halts once T falls below this value.

    max_iterations : int
        Hard cap regardless of temperature.

    reheat_enabled : bool
        When True, partially restore T whenever there is no improvement for
        `reheat_patience` consecutive iterations (up to `max_reheats` times).
        This is the primary mechanism for escaping deep local optima that
        the original Codex version lacked.

    reheat_patience : int
        Iterations without a new best before triggering a reheat.

    reheat_factor : float
        After a reheat: T ← initial_T × reheat_factor.

    max_reheats : int
        Maximum reheats per run.

    neighbour_mode : str
        Passed to generate_neighbours().
        'weighted' (default) | 'uniform' | 'swap_only'.

    candidates_per_iteration : int
        Number of feasible candidate moves sampled before each SA acceptance
        decision. Higher values improve move quality but cost more time.

    seed : int or None
        Fix for reproducibility.

    log_every : int
        Append a snapshot to result.history every N iterations. 0 = off.

    verbose : bool
        Print live progress to stdout.

    require_initial_feasible : bool
        Raise ValueError if the initial schedule has hard violations.

    enforce_candidate_feasible : bool
        Skip any candidate that fails check_all_hard() (belt-and-suspenders
        on top of the neighbour generator's own constraint checks).

    Returns
    -------
    SAResult with best_schedule, best_score, move counts, history, etc.

    Algorithm
    ---------
    ::

        T ← initial_temperature
        current, best ← csp_schedule   # feasible input from generator.py

        repeat until T < min_T or iter >= max_iter:
            candidate ← random_neighbour(current)
            if not feasible: skip
            delta ← score(candidate) − score(current)   # lower is better
            if delta ≤ 0 or random() < exp(−delta/T):
                current ← candidate
                if score(current) < score(best): best ← current
            T ← T × cooling_rate
            if stuck for reheat_patience iters:
                T ← initial_T × reheat_factor   # partial reheat

        return best
    """
    if seed is not None:
        random.seed(seed)

    if not (0 < cooling_rate < 1):
        raise ValueError("cooling_rate must be strictly between 0 and 1.")
    if initial_temperature <= 0:
        raise ValueError("initial_temperature must be positive.")
    if min_temperature < 0:
        raise ValueError("min_temperature cannot be negative.")
    if max_iterations <= 0:
        raise ValueError("max_iterations must be positive.")
    if candidates_per_iteration <= 0:
        raise ValueError("candidates_per_iteration must be positive.")

    # --- Feasibility guard on input ---
    initial_violations = check_all_hard(initial_schedule)
    if require_initial_feasible and initial_violations:
        raise ValueError(
            "The schedule passed to simulated_annealing() has hard-constraint "
            "violations. Ensure core.generator.generate_csp_schedule() produced "
            "a fully feasible schedule before calling SA.\n"
            f"First violations: {initial_violations[:5]}"
        )

    start_time = time.perf_counter()

    # --- State ---
    current_schedule = initial_schedule.copy()
    current_score    = evaluate_schedule(current_schedule)

    best_schedule  = current_schedule.copy()
    best_score     = current_score
    initial_score  = current_score
    best_iteration = 0

    temperature    = initial_temperature
    accepted_moves = 0
    rejected_moves = 0
    skipped_moves  = 0
    reheats_done   = 0
    no_improve_ctr = 0
    history: List[Dict[str, Any]] = []

    # --- Main loop ---
    iteration = 0
    while iteration < max_iterations and temperature > min_temperature:
        iteration += 1

        # 1. Pick one feasible neighbour via fast same-day swap
        swap = _random_same_day_swap(current_schedule)
        if swap is None:
            skipped_moves  += 1
            no_improve_ctr += 1
            temperature *= cooling_rate
            continue

        candidate_schedule, _ = swap
        candidate_score = evaluate_schedule(candidate_schedule)
        delta = candidate_score - current_score   # negative = improvement

        if delta <= 0 or _accept_worse_move(delta, temperature):
            current_schedule = candidate_schedule
            current_score    = candidate_score
            accepted_moves  += 1

            if current_score < best_score:
                best_schedule  = current_schedule.copy()
                best_score     = current_score
                best_iteration = iteration
                no_improve_ctr = 0
            else:
                no_improve_ctr += 1
        else:
            rejected_moves += 1
            no_improve_ctr += 1

        # 4. Geometric cooling
        temperature *= cooling_rate

        # 5. Reheat if stuck
        if (reheat_enabled
                and no_improve_ctr >= reheat_patience
                and reheats_done < max_reheats
                and temperature > min_temperature):
            temperature    = initial_temperature * reheat_factor
            no_improve_ctr = 0
            reheats_done  += 1
            if verbose:
                print(
                    f"  [SA] Reheat #{reheats_done} at iter {iteration} "
                    f"→ T={temperature:.4f}"
                )

        # 6. Log snapshot
        if log_every > 0 and iteration % log_every == 0:
            history.append({
                "iteration":     iteration,
                "temperature":   round(temperature, 6),
                "current_score": round(current_score, 4),
                "best_score":    round(best_score, 4),
                "reheats":       reheats_done,
            })
            if verbose:
                print(
                    f"  [SA] iter={iteration:>6}  T={temperature:8.4f}  "
                    f"cur={current_score:10.2f}  best={best_score:10.2f}"
                )

    elapsed = time.perf_counter() - start_time

    return SAResult(
        best_schedule=best_schedule,
        best_score=best_score,
        initial_score=initial_score,
        final_score=current_score,
        iterations=iteration,
        accepted_moves=accepted_moves,
        rejected_moves=rejected_moves,
        skipped_moves=skipped_moves,
        best_iteration=best_iteration,
        final_temperature=temperature,
        elapsed_seconds=elapsed,
        history=history,
    )


# ---------------------------------------------------------------------------
# Convenience wrapper
# ---------------------------------------------------------------------------

def run_sa(
    initial_schedule: Schedule,
    **kwargs: Any,
) -> Tuple[Schedule, float]:
    """
    Lightweight wrapper returning only (best_schedule, best_score).

    All keyword arguments are forwarded to simulated_annealing().

    Example
    -------
    >>> from core.generator import generate_csp_schedule
    >>> from core.model import load_nurses_from_csv
    >>> nurses   = load_nurses_from_csv('data/data_ai_project.csv')
    >>> schedule = generate_csp_schedule(nurses, seed=0)
    >>> best, score = run_sa(schedule, max_iterations=10_000, seed=0)
    """
    result = simulated_annealing(initial_schedule, **kwargs)
    return result.best_schedule, result.best_score


### 12. Tabu Search Algorithm (`tabu.py`)
An advanced meta-heuristic algorithm focusing heavily on Memory mechanisms. 

- Generates 60 unique swap moves, strictly picking the lowest cost (best) arrangement.
- Adds recent actions back onto a **Tabu memory Queue**, preventing the process from mathematically reverting the exact same manipulation and oscillating eternally. 
- Overrides memory blockages via Aspirations, should a forbidden sequence mathematically generate a better overall schedule state entirely.

This cell imports the required libraries and dependencies.

In [ ]:
"""Tabu search for schedule improvement.

The search assumes a near-feasible or feasible starting schedule and keeps all
hard constraints intact by only accepting day-level swaps that preserve the
daily shift counts.
"""
from __future__ import annotations

from collections import deque
from dataclasses import dataclass
import math
import random
from typing import Deque, Iterable, List, Optional, Tuple



Move = Tuple[int, int, int]


@dataclass(frozen=True)
class TabuConfig:
	iterations: int = 500
	tenure: int = 7
	neighbors_per_iteration: int = 60
	seed: Optional[int] = None
	verbose: bool = True
	max_no_improve: int = 2000
	start_from_feasible_only: bool = True


def generate_tabu_schedule(schedule: Schedule, config: TabuConfig | None = None) -> Schedule:
	"""Improve a schedule with tabu search and return the best feasible result.

	The neighborhood is a same-day swap between two nurses. This preserves each
	day's coverage counts and the one-shift-per-day structure, so the search can
	focus on score improvements without constantly rebuilding feasibility.
	"""
	config = config or TabuConfig()
	rng = random.Random(config.seed)
	current = schedule.copy()
	best = current.copy()
	best_score = _score(best)
	current_score = best_score
	tabu: Deque[Move] = deque(maxlen=max(1, config.tenure))
	no_improve = 0

	if config.start_from_feasible_only:
		violations = check_all_hard(current)
		if violations:
			raise ValueError(
				"Tabu search requires a feasible starting schedule. "
				f"Found {len(violations)} hard-constraint violations."
			)

	for iteration in range(1, config.iterations + 1):
		move, candidate, candidate_score = _best_neighbor(
			current=current,
			current_score=current_score,
			tabu=tabu,
			rng=rng,
			neighbors_per_iteration=config.neighbors_per_iteration,
		)

		if move is None or candidate is None:
			continue

		current = candidate
		current_score = candidate_score
		tabu.append(move)

		if candidate_score + 1e-9 < best_score:
			best = candidate.copy()
			best_score = candidate_score
			no_improve = 0
		else:
			no_improve += 1

		if config.verbose and iteration % 25 == 0:
			print(
				f"[tabu] iter={iteration} current={current_score:.2f} "
				f"best={best_score:.2f} tabu={len(tabu)}"
			)

		if no_improve >= config.max_no_improve:
			break

	return best


def _score(schedule: Schedule) -> float:
	"""Score feasible schedules directly; heavily penalize infeasible ones."""
	violations = check_all_hard(schedule)
	if not violations:
		return evaluate_schedule(schedule)
	return evaluate_schedule(schedule) + 10000.0 * len(violations)

def _best_neighbor(
  current: Schedule,
  current_score: float,
  tabu: Deque[Move],
  rng: random.Random,
  neighbors_per_iteration: int,
) -> Tuple[Optional[Move], Optional[Schedule], float]:
  """Explore a limited set of same-day swaps and return the best admissible one."""
  all_moves = list(_candidate_moves(current))
  if not all_moves:
    return None, None, math.inf

  if len(all_moves) > neighbors_per_iteration:
    moves = rng.sample(all_moves, neighbors_per_iteration)
  else:
    moves = all_moves

  best_move: Optional[Move] = None
  best_sched: Optional[Schedule] = None
  best_score = math.inf
  best_is_tabu = False

  for move in moves:
    candidate = _apply_swap(current, move)
    if candidate is None:
      continue
    if check_all_hard(candidate):
      continue

    score = evaluate_schedule(candidate)
    is_tabu = move in tabu
    if is_tabu and score >= current_score - 1e-9:
      continue

    if (
      score < best_score - 1e-9
      or (abs(score - best_score) <= 1e-9 and best_is_tabu and not is_tabu)
    ):
      best_move = move
      best_sched = candidate
      best_score = score
      best_is_tabu = is_tabu

  return best_move, best_sched, best_score


def _candidate_moves(schedule: Schedule) -> Iterable[Move]:
  """Yield unique day-level swap moves (day, nurse_a, nurse_b)."""
  for day in range(1, NUM_DAYS + 1):
    for idx, nurse_a in enumerate(schedule.nurses):
      for nurse_b in schedule.nurses[idx + 1 :]:
        a = schedule.get(day, nurse_a.nurse_id)
        b = schedule.get(day, nurse_b.nurse_id)
        if a == b:
          continue
        yield (day, nurse_a.nurse_id, nurse_b.nurse_id)


def _apply_swap(schedule: Schedule, move: Move) -> Optional[Schedule]:
  """Swap the two nurses' assignments on the given day and return a copy."""
  day, nurse_id_a, nurse_id_b = move
  shift_a = schedule.get(day, nurse_id_a)
  shift_b = schedule.get(day, nurse_id_b)
  if shift_a == shift_b:
    return None

  candidate = schedule.copy()
  candidate.set(day, nurse_id_a, shift_b)
  candidate.set(day, nurse_id_b, shift_a)
  return candidate

---
## Tests

### Test 1 — Generator

This cell implements the CSP and backtracking algorithm.

In [ ]:
nurses = load_nurses_from_csv(CSV_PATH)
print(f'Loaded {len(nurses)} nurses, {sum(1 for n in nurses if n.is_senior)} seniors')

csp_schedule = generate_schedule(nurses, seed=None)
violations = check_all_hard(csp_schedule)
csp_score = evaluate_schedule(csp_schedule)
print(f'Hard violations : {len(violations)}')
print(f'Soft score      : {csp_score:.2f}')
print(csp_schedule.summary())


### Test 2 — Greedy

This cell displays scheduling results and outputs.

In [ ]:
nurses = load_nurses_from_csv(CSV_PATH)
greedy_schedule = generate_greedy_schedule(Schedule(nurses))
violations = check_all_hard(greedy_schedule)
greedy_score = evaluate_schedule(greedy_schedule)
print(f'Hard violations : {len(violations)}')
print(f'Soft score      : {greedy_score:.2f}')
print(greedy_schedule.summary())


### Test 3 — Generator → Tabu Search

This cell displays scheduling results and outputs.

In [20]:
nurses = load_nurses_from_csv(CSV_PATH)
tabu_initial = generate_schedule(nurses, seed=1)
print(f'Initial: violations={len(check_all_hard(tabu_initial))}, score={evaluate_schedule(tabu_initial):.2f}')

# Collect score history every 25 iterations for convergence plot
tabu_history = []
_tabu_nurses = nurses
_tabu_sched  = tabu_initial.copy()
_tabu_best   = _tabu_sched.copy()
_tabu_best_score = evaluate_schedule(_tabu_best)
_tabu_score  = _tabu_best_score
tabu_history.append(_tabu_best_score)

config = TabuConfig(iterations=2700, seed=None, verbose=True,
                    neighbors_per_iteration=40, max_no_improve=200)
tabu_schedule = generate_tabu_schedule(tabu_initial, config)
tabu_score = evaluate_schedule(tabu_schedule)
print(f'After Tabu: violations={len(check_all_hard(tabu_schedule))}, score={tabu_score:.2f}')
print(tabu_schedule.summary())


[tabu] iter=100 current=1258.62 best=1258.62 tabu=7
[tabu] iter=125 current=1235.82 best=1235.82 tabu=7
[tabu] iter=150 current=1153.70 best=1153.70 tabu=7
[tabu] iter=175 current=1099.95 best=1099.95 tabu=7
[tabu] iter=200 current=1064.99 best=1064.99 tabu=7
[tabu] iter=225 current=1030.82 best=1029.99 tabu=7
[tabu] iter=250 current=922.87 best=922.87 tabu=7
[tabu] iter=275 current=750.79 best=750.79 tabu=7
[tabu] iter=300 current=695.29 best=695.29 tabu=7
[tabu] iter=325 current=741.79 best=682.95 tabu=7
[tabu] iter=350 current=684.12 best=682.95 tabu=7
[tabu] iter=375 current=669.14 best=660.97 tabu=7
[tabu] iter=400 current=593.24 best=593.24 tabu=7
[tabu] iter=425 current=592.87 best=592.87 tabu=7
[tabu] iter=450 current=572.04 best=572.04 tabu=7
[tabu] iter=475 current=534.65 best=534.65 tabu=7
[tabu] iter=500 current=502.82 best=470.65 tabu=7
[tabu] iter=525 current=422.40 best=422.40 tabu=7
[tabu] iter=550 current=279.24 best=279.24 tabu=7
[tabu] iter=575 current=251.87 best=25

KeyboardInterrupt: 

### Test 4 — Generator → Simulated Annealing

This cell displays scheduling results and outputs.

In [ ]:
nurses = load_nurses_from_csv(CSV_PATH)
sa_initial = generate_schedule(nurses, seed=0)
print(f'Initial: violations={len(check_all_hard(sa_initial))}, score={evaluate_schedule(sa_initial):.2f}')

sa_result = simulated_annealing(
    sa_initial,
    initial_temperature=150.0,
    cooling_rate=0.999359,
    min_temperature=0.01,
    max_iterations=5000,
    reheat_enabled=True,
    reheat_patience=500,
    reheat_factor=0.25,
    max_reheats=5,
    seed=0,
    verbose=True,
    log_every=100,
)
sa_schedule = sa_result.best_schedule
sa_score    = sa_result.best_score
print(sa_result.summary())
print(f'Hard violations in best: {len(check_all_hard(sa_schedule))}')
print(sa_schedule.summary())


---
## Visualizations

Run all Test cells (1–4) before running these.

**Charts included:**
1. Color-coded roster view (CSP, Greedy, Tabu, SA)
2. Hours worked per nurse — fairness bar chart
3. SA vs Tabu convergence comparison
4. Greedy vs CSP monthly-hour constraint analysis

### 1 — Color-Coded Roster View
Shows the full 28-day schedule per nurse for all 4 algorithms.

This cell imports the required libraries and dependencies.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

SHIFT_COLORS = {'D': '#4CAF50', 'L': '#2196F3', 'N': '#9C27B0', 'O': '#EEEEEE', 'F': '#FF9800'}
SHIFT_LABELS = {'D': 'Day', 'L': 'Late', 'N': 'Night', 'O': 'Off', 'F': 'Day-off'}

def plot_roster(schedule, title, ax):
    n_nurses = len(schedule.nurses)
    n_days   = NUM_DAYS
    grid = np.zeros((n_nurses, n_days, 3))

    nurse_labels = []
    for row, nurse in enumerate(schedule.nurses):
        tag = "[SR]" if nurse.is_senior else "[JR]"
        nurse_labels.append(f"{tag} {nurse.first_name} {nurse.last_name}")
        for day in range(1, n_days + 1):
            shift = schedule.get(day, nurse.nurse_id)
            col   = matplotlib.colors.to_rgb(SHIFT_COLORS.get(shift, '#FFFFFF'))
            grid[row, day - 1] = col

    ax.imshow(grid, aspect='auto', interpolation='nearest')
    ax.set_title(title, fontsize=11, fontweight='bold', pad=8)
    ax.set_xlabel('Day', fontsize=9)
    ax.set_yticks(range(n_nurses))
    ax.set_yticklabels(nurse_labels, fontsize=6.5)
    ax.set_xticks(range(0, n_days, 7))
    ax.set_xticklabels([f"W{i//7+1}" for i in range(0, n_days, 7)], fontsize=8)
    # day grid lines
    for d in range(n_days):
        ax.axvline(d - 0.5, color='white', linewidth=0.3)
    for n in range(n_nurses):
        ax.axhline(n - 0.5, color='white', linewidth=0.3)

fig, axes = plt.subplots(2, 2, figsize=(20, 14))
fig.suptitle('Color-Coded Roster View — All Algorithms', fontsize=14, fontweight='bold', y=1.01)

plot_roster(csp_schedule,    f'CSP Generator  (score={csp_score:.0f})',    axes[0][0])
plot_roster(greedy_schedule, f'Greedy         (score={greedy_score:.0f})', axes[0][1])
plot_roster(tabu_schedule,   f'Tabu Search    (score={tabu_score:.0f})',   axes[1][0])
plot_roster(sa_schedule,     f'Simulated Annealing (score={sa_score:.0f})', axes[1][1])

# Legend
patches = [mpatches.Patch(color=c, label=SHIFT_LABELS[s]) for s, c in SHIFT_COLORS.items()]
fig.legend(handles=patches, loc='lower center', ncol=5, fontsize=10,
           bbox_to_anchor=(0.5, -0.03), frameon=True)

plt.tight_layout()
plt.savefig('roster_view.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: roster_view.png")


### 2 — Fairness Chart: Hours Worked per Nurse
Bar chart per algorithm. Red = senior, blue = junior. Dashed lines show min/max hard constraint bounds.

This cell imports the required libraries and dependencies.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 10))
fig.suptitle('Fairness Chart — Hours Worked per Nurse', fontsize=13, fontweight='bold')

def plot_hours(schedule, title, ax, score):
    names  = [f"{n.first_name[:8]} {n.last_name[:8]}" for n in schedule.nurses]
    hours  = [schedule.total_hours(n.nurse_id) for n in schedule.nurses]
    colors = ['#E74C3C' if n.is_senior else '#3498DB' for n in schedule.nurses]

    bars = ax.bar(range(len(names)), hours, color=colors, edgecolor='white', linewidth=0.5)
    ax.axhline(sum(hours)/len(hours), color='black', linewidth=1.5, linestyle='--',
               label=f'Mean: {sum(hours)/len(hours):.0f}h')
    ax.axhline(HARD.get("min_monthly_hours", 120), color='red', linewidth=1,
               linestyle=':', label=f'Min: {HARD.get("min_monthly_hours",120)}h')
    ax.axhline(HARD.get("max_monthly_hours", 170), color='orange', linewidth=1,
               linestyle=':', label=f'Max: {HARD.get("max_monthly_hours",170)}h')

    ax.set_title(f"{title}  (score={score:.0f})", fontsize=10, fontweight='bold')
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=45, ha='right', fontsize=6.5)
    ax.set_ylabel('Hours', fontsize=9)
    ax.set_ylim(0, max(hours) + 20)
    ax.legend(fontsize=7)

    # annotate hours on bars
    for bar, h in zip(bars, hours):
        ax.text(bar.get_x() + bar.get_width()/2, h + 1, str(h),
                ha='center', va='bottom', fontsize=5.5, rotation=90)

    import statistics
    std = statistics.stdev(hours) if len(hours) > 1 else 0
    ax.set_xlabel(f'std={std:.1f}h  |  min={min(hours)}h  max={max(hours)}h', fontsize=8)

    sr_patch = mpatches.Patch(color='#E74C3C', label='Senior')
    jr_patch = mpatches.Patch(color='#3498DB', label='Junior')
    ax.legend(handles=ax.get_legend_handles_labels()[0] + [sr_patch, jr_patch],
              labels=ax.get_legend_handles_labels()[1] + ['Senior', 'Junior'],
              fontsize=6.5, loc='upper right')

plot_hours(csp_schedule,    'CSP Generator',       axes[0][0], csp_score)
plot_hours(greedy_schedule, 'Greedy',              axes[0][1], greedy_score)
plot_hours(tabu_schedule,   'Tabu Search',         axes[1][0], tabu_score)
plot_hours(sa_schedule,     'Simulated Annealing', axes[1][1], sa_score)

plt.tight_layout()
plt.savefig('fairness_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fairness_chart.png")


### 3 — Convergence: SA vs Tabu Search
Shows how score improves over iterations for both algorithms.

This cell imports the required libraries and dependencies.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

# SA history comes from sa_result.history (logged every 100 iters)
sa_iters  = [h['iteration']  for h in sa_result.history]
sa_bests  = [h['best_score'] for h in sa_result.history]
sa_currs  = [h['current_score'] for h in sa_result.history]

ax.plot(sa_iters, sa_currs, color='#3498DB', alpha=0.35, linewidth=1,   label='SA current score')
ax.plot(sa_iters, sa_bests, color='#2980B9', linewidth=2,               label='SA best score')

# Tabu: generate_tabu_schedule doesn't return history, so we re-run briefly
# with verbose=False and collect scores by monkey-patching evaluate_schedule
_tabu_scores = []
_tabu_iters  = []

class _TabuTracker:
    def __init__(self, inner):
        self._inner = inner
        self._call  = 0
    def __call__(self, sched):
        self._call += 1
        score = self._inner(sched)
        _tabu_scores.append(score)
        _tabu_iters.append(self._call)
        return score

_orig_eval = evaluate_schedule
_tracker   = _TabuTracker(_orig_eval)

import builtins
_g = globals()
_g['evaluate_schedule'] = _tracker
_tabu_init2 = generate_schedule(nurses, seed=1)
_tabu_cfg2  = TabuConfig(iterations=500, seed=1, verbose=False,
                          neighbors_per_iteration=40, max_no_improve=100)
_tabu_sched2 = generate_tabu_schedule(_tabu_init2, _tabu_cfg2)
_g['evaluate_schedule'] = _orig_eval

# compute running best from tabu scores
_tabu_best_curve = []
_cur_best = float('inf')
for s in _tabu_scores:
    if s < _cur_best: _cur_best = s
    _tabu_best_curve.append(_cur_best)

if _tabu_iters:
    ax.plot(_tabu_iters, _tabu_scores,     color='#E74C3C', alpha=0.35, linewidth=1, label='Tabu current score')
    ax.plot(_tabu_iters, _tabu_best_curve, color='#C0392B', linewidth=2,             label='Tabu best score')

ax.axhline(0, color='grey', linewidth=0.8, linestyle='--', alpha=0.5)
ax.set_xlabel('Iteration', fontsize=11)
ax.set_ylabel('Soft Score (lower = better)', fontsize=11)
ax.set_title('Convergence: Simulated Annealing vs Tabu Search', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('convergence.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: convergence.png")


### 4 — Greedy vs CSP: Monthly Hour Constraint Analysis
Shows how many nurses violate the min/max hour bounds under each approach.

This cell implements functions used for scheduling operations.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Greedy vs CSP — Monthly Hour Constraint Analysis', fontsize=12, fontweight='bold')

min_h = HARD.get("min_monthly_hours", 120)
max_h = HARD.get("max_monthly_hours", 170)

def plot_hour_dist(schedule, title, ax, color):
    hours = [schedule.total_hours(n.nurse_id) for n in schedule.nurses]
    names = [f"{n.first_name[:6]}.{n.last_name[:6]}" for n in schedule.nurses]

    violations_min = sum(1 for h in hours if h < min_h)
    violations_max = sum(1 for h in hours if h > max_h)

    bar_colors = []
    for h in hours:
        if h < min_h:   bar_colors.append('#E74C3C')   # red = too few
        elif h > max_h: bar_colors.append('#E67E22')   # orange = too many
        else:           bar_colors.append(color)        # ok

    ax.bar(range(len(names)), hours, color=bar_colors, edgecolor='white')
    ax.axhspan(min_h, max_h, alpha=0.1, color='green', label=f'Valid range [{min_h}–{max_h}h]')
    ax.axhline(min_h, color='red',    linewidth=1.5, linestyle='--', label=f'Min {min_h}h')
    ax.axhline(max_h, color='orange', linewidth=1.5, linestyle='--', label=f'Max {max_h}h')
    ax.axhline(sum(hours)/len(hours), color='black', linewidth=1.5, linestyle='-',
               label=f'Mean {sum(hours)/len(hours):.0f}h')

    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=45, ha='right', fontsize=7)
    ax.set_ylabel('Hours worked', fontsize=10)
    ax.set_ylim(0, max(hours) + 30)
    ax.set_title(
        f"{title}\n"
        f"Under min: {violations_min} nurses  |  Over max: {violations_max} nurses  |  "
        f"Score: {evaluate_schedule(schedule):.0f}",
        fontsize=9, fontweight='bold'
    )
    ax.legend(fontsize=8)

    red_p    = mpatches.Patch(color='#E74C3C', label=f'Below min ({violations_min})')
    orange_p = mpatches.Patch(color='#E67E22', label=f'Above max ({violations_max})')
    ok_p     = mpatches.Patch(color=color,     label=f'Within range ({len(hours)-violations_min-violations_max})')
    ax.legend(handles=[red_p, orange_p, ok_p,
                        plt.Line2D([0],[0],color='red',linestyle='--',label=f'Min {min_h}h'),
                        plt.Line2D([0],[0],color='orange',linestyle='--',label=f'Max {max_h}h'),
                        plt.Line2D([0],[0],color='black',label=f'Mean {sum(hours)/len(hours):.0f}h')],
              fontsize=7.5)

plot_hour_dist(greedy_schedule, 'Greedy Scheduler', axes[0], '#85C1E9')
plot_hour_dist(csp_schedule,    'CSP Generator',    axes[1], '#82E0AA')

plt.tight_layout()
plt.savefig('greedy_vs_csp.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: greedy_vs_csp.png")


### 5 — Algorithm Summary
Side-by-side soft score and fairness (hour std dev) for all 4 algorithms.

This cell imports the required libraries and dependencies.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Algorithm Performance Summary', fontsize=13, fontweight='bold')

algos  = ['CSP Generator', 'Greedy', 'Tabu Search', 'SA']
scores = [csp_score, greedy_score, tabu_score, sa_score]
colors = ['#5DADE2', '#58D68D', '#E74C3C', '#F39C12']

# Soft score comparison
bars = axes[0].bar(algos, scores, color=colors, edgecolor='white', linewidth=0.8)
axes[0].axhline(0, color='black', linewidth=1, linestyle='--', alpha=0.5)
axes[0].set_ylabel('Soft Score (lower = better)', fontsize=10)
axes[0].set_title('Final Soft Score per Algorithm', fontsize=10, fontweight='bold')
for bar, score in zip(bars, scores):
    y = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 y + (5 if y >= 0 else -25),
                 f"{score:.0f}", ha='center', fontsize=9, fontweight='bold')

# Hours std deviation (fairness)
stds = []
import statistics
for sched in [csp_schedule, greedy_schedule, tabu_schedule, sa_schedule]:
    hrs = [sched.total_hours(n.nurse_id) for n in sched.nurses]
    stds.append(statistics.stdev(hrs) if len(hrs) > 1 else 0)

bars2 = axes[1].bar(algos, stds, color=colors, edgecolor='white', linewidth=0.8)
axes[1].set_ylabel('Std Dev of Hours (lower = fairer)', fontsize=10)
axes[1].set_title('Fairness: Hour Distribution Std Dev', fontsize=10, fontweight='bold')
for bar, std in zip(bars2, stds):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f"{std:.1f}h", ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('summary.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: summary.png")
